In [ ]:
# CELL 1: Setup PATH and Imports
import os
import subprocess
import numpy as np
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr,kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity 
import pandas as pd
# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']

In [ ]:
fmri_data = fmri_right_processed
neighbor_indices = np.load('/home/amin/Research/Representation/Movie/outputs/searchlight_rsa_output/left_hemisphere/neighbor_indices_k100.npy')
# fMRI data shape
n_vertices, n_timepoints = fmri_data.shape

model_rdm = load_model_rdms(model_file)

print(f"Model RDM shape: {model_rdm.shape}")

# Verify model RDM matches timepoints
if model_rdm.shape[0] != n_timepoints:
    raise ValueError(
        f"Model RDM timepoints ({model_rdm.shape[0]}) doesn't match "
        f"fMRI timepoints ({n_timepoints})"
    )

# Vectorize model RDM once (upper triangle)
triu_indices = np.triu_indices(n_timepoints, k=1)
model_rdm_vec = model_rdm[triu_indices]

# Initialize output arrays
n_searchlights = neighbor_indices.shape[0]
rsa_correlations = np.zeros(n_searchlights, dtype=np.float32)
rsa_pvalues = np.zeros(n_searchlights, dtype=np.float32)

method = 'spearman'

print(f"\nComputing RSA for {n_searchlights} searchlights...")
print(f"Method: {method}")


# Main loop - compute RSA for each searchlight
for vertex_idx in range(n_searchlights):
    if vertex_idx % 10 == 0:
        print(f"Progress: {vertex_idx}/{n_searchlights} ({100*vertex_idx/n_searchlights:.1f}%)")
    
    neighbors = neighbor_indices[vertex_idx, :]
    searchlight_activations = fmri_data[neighbors, :].T  # (n_timepoints, k_neighbors)
    

    searchlight_rdm = squareform(pdist(searchlight_activations, metric='correlation'))
    
    # Vectorize searchlight RDM
    searchlight_rdm_vec = searchlight_rdm[triu_indices]
    
    # Compute correlation with model RDM
    if method == 'spearman':
        corr, pval = spearmanr(searchlight_rdm_vec, model_rdm_vec)
    elif method == 'pearson':
        corr, pval = pearsonr(searchlight_rdm_vec, model_rdm_vec)
    elif method == 'kendall':
        corr, pval = kendalltau(searchlight_rdm_vec, model_rdm_vec)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Store results
    rsa_correlations[vertex_idx] = corr
    rsa_pvalues[vertex_idx] = pval

print(f"\nRSA computation complete!")
print(f"Correlation range: [{np.nanmin(rsa_correlations):.3f}, {np.nanmax(rsa_correlations):.3f}]")
print(f"Mean correlation: {np.nanmean(rsa_correlations):.3f}")

In [ ]:
vertex_idx = 1234
neighbors = neighbor_indices[vertex_idx, :]
searchlight_activations = fmri_data[neighbors, :].T  # (n_timepoints, k_neighbors)

cos_sim = cosine_similarity(searchlight_activations)
searchlight_rdm_cosine = 1 - cos_sim
searchlight_rdm_vec_cosine = searchlight_rdm_cosine[triu_indices]

searchlight_rdm_corr = squareform(pdist(searchlight_activations, metric='correlation'))
searchlight_rdm_vec_corr = searchlight_rdm_corr[triu_indices]

corr, pval = spearmanr(searchlight_rdm_vec_corr, model_rdm_vec)
print(f"CORRELATION RDM: corr={corr:.4f}, pval={pval:.4e}")
corr, pval = spearmanr(searchlight_rdm_vec_cosine, model_rdm_vec)
print(f"COSINE RDM: corr={corr:.4f}, pval={pval:.4e}")

# # Compute correlation with model RDM
# if method == 'spearman':
#     corr, pval = spearmanr(searchlight_rdm_vec, model_rdm_vec)
# elif method == 'pearson':
#     corr, pval = pearsonr(searchlight_rdm_vec, model_rdm_vec)
# elif method == 'kendall':
#     corr, pval = kendalltau(searchlight_rdm_vec, model_rdm_vec)
# else:
#     raise ValueError(f"Unknown method: {method}")

# # Store results
# rsa_correlations[vertex_idx] = corr
# rsa_pvalues[vertex_idx] = pval